
#  Global Disease Outbreak Visualization System

**Domain:** Public Health Data Science & Geospatial Analytics


## 📌 Project Overview

This project is an interactive, data-driven visualization system engineered to track, analyze, and animate the global transmission dynamics of infectious disease outbreaks. By combining robust data engineering pipelines with modern geospatial and time-series graphing engines, this system converts raw public health logs into an intuitive, visual intelligence dashboard.

The primary focus of this implementation was cleaning raw time-series data, resolving geographic inconsistencies, and developing interactive visual assets to map pandemic velocity over time.


## 🚀 Core Features Developed

* **Animated Global Choropleth Timeline:** A dynamic world map that tracks and plays back the macroeconomic geographic spread of disease transmission across chronological thresholds.
* **Epidemic Wave Velocity Vectoring:** A time-series curve metric visualizing 7-day smoothed running averages to track real daily case momentum instead of purely flat cumulative numbers.
* **Territorial Grouping Data Cleanse:** Operational logic handling that automatically aggregates granular local states, provinces, and dependencies into unified national totals.
* **Algorithmic ISO Mapping:** Dictionary-based text preprocessing that dynamically matches text-based country variations into standardized ISO Alpha-3 geographic coordinates for flawless map rendering.



## 🛠️ System Architecture & Engineering Stack

The system processes data linearly through a specialized ingestion and rendering pipeline:


[Kaggle API / Data Source]
       │
       ▼
[Pandas Pipeline] ──► (Group Provinces -> Convert Naming to ISO Alpha-3)
       │
       ▼
[Feature Engineering] ──► (Calculate .diff() for New Cases -> 7-Day Rolling Mean)
       │
       ▼
[Plotly Graphics Engine] ──► (Animated Global Map & Wave Velocity Line Charts)



* **Data Pipeline & Extraction:** `kagglehub API` (Automated data fetching directly from online public data repositories).
* **Data Wrangling & Time-Series Math:** `pandas` & `numpy` (Regional rolling windows, delta difference computations, and row aggregations).
* **Geospatial Translation Layer:** `country_converter` (Text-to-ISO3 coordinate cleaning engine).
* **Visual Rendering Engine:** `plotly.express` (Animated spatial-temporal layers and unified hover analytics).



## 📊 Key Metrics Tracked

1. **Cumulative Confirmed Cases:** Total historical footprint of the outbreak per region.
2. **Daily New Cases ($\Delta$ Cases):** Calculated using `.diff()` to show the active, daily acceleration of the disease.
3. **7-Day Moving Average:** Applied using `.rolling().mean()` to smooth out weekend reporting anomalies and reveal the true epidemic curve.



In [ ]:
 !pip install kagglehub country_converter plotly pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 2.6 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import country_converter as coco

# 1. Log into Kaggle (Use your custom credentials)
os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"
os.environ["KAGGLE_KEY"] = "your_abc123_api_key"

print("Fetching data from Kaggle...")
# 2. Retrieve data directly into a Pandas DataFrame
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    handle="imdevskp/corona-virus-report",
    path="covid_19_clean_complete.csv"
)

# 3. Clean up formatting (Convert the Date column into strings sorted chronologically)
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')
df = df.sort_values('Date')

# 4. Generate ISO Alpha-3 Country codes automatically
print("Converting country names to ISO3 codes (This might take a moment)...")
cc = coco.CountryConverter()
df['Country_Code'] = cc.pandas_convert(series=df['Country/Region'], to='ISO3')

print("\nData Pipeline Complete! Preview:")
print(df[['Date', 'Country/Region', 'Country_Code', 'Confirmed', 'Deaths']].head())

Fetching data from Kaggle...
Using Colab cache for faster access to the 'corona-virus-report' dataset.
Converting country names to ISO3 codes (This might take a moment)...

Data Pipeline Complete! Preview:
           Date Country/Region Country_Code  Confirmed  Deaths
0    2020-01-22    Afghanistan          AFG          0       0
166  2020-01-22    Netherlands          NLD          0       0
167  2020-01-22    Netherlands          NLD          0       0
168  2020-01-22    New Zealand          NZL          0       0
169  2020-01-22      Nicaragua          NIC          0       0


In [ ]:
import plotly.express as px

print("Generating map visualization...")

fig_map = px.choropleth(
    df,
    locations="Country_Code",         # Uses the ISO3 column we added in Step 2
    color="Confirmed",                # Shades the map based on the Confirmed cases count
    hover_name="Country/Region",      # Displays country name on mouse hover
    animation_frame="Date",           # Binds data points across the chronological slider
    color_continuous_scale="Reds",    # Red color gradient palette
    range_color=[0, df['Confirmed'].max() * 0.05], # Caps scale max to 5% of max peak so early spread is visible
    title="Global Viral Outbreak Spread Timeline"
)

# Optimize design layout for an immersive visual experience
fig_map.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type='equirectangular'),
    width=1000,
    height=600
)

# Render live chart directly in your Colab notebook
fig_map.show()

Generating map visualization...


In [ ]:
# Group by Date and Country to sum up all provincial data into a single national total
df_clean = df.groupby(['Date', 'Country/Region', 'Country_Code'])[['Confirmed', 'Deaths']].sum().reset_index()

# View the truly clean, aggregated preview
print(df_clean[['Date', 'Country/Region', 'Country_Code', 'Confirmed', 'Deaths']].head())

         Date Country/Region Country_Code  Confirmed  Deaths
0  2020-01-22    Afghanistan          AFG          0       0
1  2020-01-22        Albania          ALB          0       0
2  2020-01-22        Algeria          DZA          0       0
3  2020-01-22        Andorra          AND          0       0
4  2020-01-22         Angola          AGO          0       0


In [ ]:
import pandas as pd
import numpy as np

def process_backend_data(df_raw):
    # 1. Clean Dates & Group Territories (combining states/provinces)
    df_raw['Date'] = pd.to_datetime(df_raw['Date'])
    df_grouped = df_raw.groupby(['Date', 'Country/Region', 'Country_Code']).agg({
        'Confirmed': 'sum',
        'Deaths': 'sum'
    }).reset_index()

    # Sort chronologically by country to safely perform time-series math
    df_grouped = df_grouped.sort_values(by=['Country/Region', 'Date'])

    # 2. Calculate Daily New Cases using .diff()
    # We group by country so Day 1 of Country B doesn't subtract Day Last of Country A
    df_grouped['New_Cases'] = df_grouped.groupby('Country/Region')['Confirmed'].diff().fillna(0)
    df_grouped['New_Deaths'] = df_grouped.groupby('Country/Region')['Deaths'].diff().fillna(0)

    # Fix any potential data reporting drops (negative cases aren't epidemiologically real)
    df_grouped['New_Cases'] = df_grouped['New_Cases'].clip(lower=0)
    df_grouped['New_Deaths'] = df_grouped['New_Deaths'].clip(lower=0)

    # 3. Calculate 7-Day Rolling Averages to smooth out weekend reporting lag
    df_grouped['New_Cases_7Day_Avg'] = df_grouped.groupby('Country/Region')['New_Cases'].transform(lambda x: x.rolling(7, min_periods=1).mean())

    # 4. Calculate Mortality Rate
    df_grouped['Mortality_Rate_%'] = np.where(df_grouped['Confirmed'] > 0, (df_grouped['Deaths'] / df_grouped['Confirmed']) * 100, 0)

    return df_grouped

# Test the backend function (assuming df is loaded from your previous notebook step)
clean_df = process_backend_data(df)
print(clean_df.tail())

            Date Country/Region Country_Code  Confirmed  Deaths  New_Cases  \
34407 2020-07-23       Zimbabwe          ZWE       2124      28       90.0   
34594 2020-07-24       Zimbabwe          ZWE       2296      32      172.0   
34781 2020-07-25       Zimbabwe          ZWE       2434      34      138.0   
34968 2020-07-26       Zimbabwe          ZWE       2512      34       78.0   
35155 2020-07-27       Zimbabwe          ZWE       2704      36      192.0   

       New_Deaths  New_Cases_7Day_Avg  Mortality_Rate_%  
34407         2.0          108.857143          1.318267  
34594         4.0          125.142857          1.393728  
34781         2.0          136.571429          1.396878  
34968         0.0          128.714286          1.353503  
35155         2.0          141.571429          1.331361  
